# 2 · The tool layer

Layer 1. Exact arguments in, real rows out. No judgement is made here — these
report what happened and leave what it means to the specialist above them.

17 tools across four domains, and the sets are **pairwise disjoint**. That is
easy to claim and easy to lose the first time a fifth tool looks useful in two
places, so it is asserted rather than trusted.

In [1]:
# Reload the package from disk on every run, so an edit to src/sentinel takes
# effect without restarting the kernel. Python caches imported modules in
# sys.modules and a stale one will happily report yesterday's numbers.
import sys, pathlib
for name in [m for m in sys.modules if m.startswith("sentinel")]:
    del sys.modules[name]

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("sentinel package:", ROOT / "src" / "sentinel")

sentinel package: D:\AgentBuilder2026datasense\Sentinel-MultiagentFraudAnalyst\src\sentinel


In [2]:
import sentinel.tools as T

for domain, tools in T.TOOLSETS.items():
    print(f"{domain:<12} {len(tools)} tools")
    for t in tools:
        print(f"    {t.name}")
    print()

print("isolation violations:", T.check_isolation() or "none")

behaviour    7 tools
    get_alerts
    get_incident_activity
    get_spending_baseline
    get_device_history
    get_geography
    get_high_risk_merchant_activity
    get_limit_utilisation

context      4 tools
    get_customer_profile
    get_case_notes
    get_disputes
    get_prior_cases

network      3 tools
    get_shared_devices
    get_device_peers
    get_merchant_overlap

disposition  3 tools
    record_disposition
    block_card
    escalate_case

isolation violations: none


## The disposition officer holds no read tool

It writes; it does not read. So it cannot quietly look something up to patch a
gap in what it was told — it has to decide on the findings it was handed. That
is what makes the supervisor's routing order mean something.

In [3]:
read = {t.name for t in T.READ_TOOLS}
write = {t.name for t in T.DISPOSITION_TOOLS}
print("read tools :", len(read))
print("write tools:", sorted(write))
print("overlap    :", read & write or "none")

read tools : 14
write tools: ['block_card', 'escalate_case', 'record_disposition']
overlap    : none


## What a tool actually returns

Formatted text, not JSON. Across 276 accounts and four specialists an aligned
table costs meaningfully fewer tokens than a nested object, and a model reads it
at least as well.

In [4]:
from sentinel.tools.behaviour_tools import get_alerts, get_incident_activity
print(get_alerts.invoke({"account_id": "A00985"}))

1 alert(s) on A00985:

  AL0170  R02 New device high value  [high]  fired 2026-02-27T12:46:44
    rule looks for : Transaction above 25,000 from a device first seen in the last 24 hours.
    trigger txn    : T0107306 at 2026-02-27T15:46:44, 66,340, app, ip=IN, merchant=Travel and airlines 151 (travel)

Note: the trigger transaction often happens AFTER triggered_at. Use get_incident_activity for the full episode.


In [5]:
print(get_incident_activity.invoke({"account_id": "A00985"}))

Incident window 2026-02-27T10:46:44 -> 2026-02-27T17:46:44
4 transactions (4 approved), total 216,091

txn_id     timestamp                amount ch            ip  auth      device   merchant
T0107303   2026-02-27T12:46:44      36,861 app           IN  approved  DX01444  Grocery 271 (grocery, risk 0.04)
T0107304   2026-02-27T13:46:44      64,146 app           IN  approved  DX01444  Online retail 167 (ecommerce, risk 0.3)
T0107305   2026-02-27T14:46:44      48,745 app           IN  approved  DX01444  Utilities 370 (utilities, risk 0.14)
T0107306   2026-02-27T15:46:44      66,340 app           IN  approved  DX01444  Travel and airlines 151 (travel, risk 0.27)


## Silence is a finding

An account with nothing on file does not get an empty list. It gets a sentence
saying so, and what that implies.

In [6]:
from sentinel.tools.context_tools import get_case_notes, get_prior_cases
print(get_prior_cases.invoke({"account_id": "A00985"}))
print()
print(get_case_notes.invoke({"account_id": "A00985"}))

(no prior investigations — this customer has not been reviewed before)

1 case note(s):

  N00080  2026-02-27T07:46:44  [before_incident]  0.1 days before the incident
    author  : T. Fernandes via branch
    note    : "Support chat. Customer upgraded their phone on the 14th and could not log in. Walked them through re-registration. Verified with video KYC."


## Citations are checked before anything is written

Three layers: the identifier has the right **shape**, the row **exists and
belongs to this account**, and any **quoted words** appear in the stored text.

`AL0001` is a perfectly valid alert id. It belongs to a different account.

In [7]:
from sentinel import db, validation
db.init_runtime()

print(validation.check_shape("alert", "ALxxxx1"))
print()
print(validation.check_ownership("note", "N99999", "A00985"))
print()
print(validation.check_quote("note", "N00080", "the customer admitted everything"))

'ALxxxx1' is not a valid alert id. Expected the form AL\d{4}, for example AL0170. Do not invent or abbreviate identifiers.

note N99999 does not exist for account A00985. Cite only records returned by your own tool calls.

The quoted words are not in note N00080. Quote the stored text exactly, or cite the id without a quote.


In [8]:
# And the rule that a `legitimate` verdict cannot rest on numbers alone.
d = validation.Disposition(
    account_id="A00985", verdict="legitimate", confidence="high",
    reasoning="The spending is large but the geography and timing are ordinary for this customer.",
    evidence=[validation.EvidenceRef("alert", "AL0170")],
)
for problem in validation.validate(d):
    print("-", problem)

- reasoning is too short to be defensible. State what fired, what the evidence said, and why it settles the question.
- a 'legitimate' verdict must cite something a human wrote - a case note, a dispute statement or a prior case. Numbers alone cannot explain an alert away. If the file holds no such record, the honest verdict is 'insufficient_evidence'.
